# Support Vector Machines dan Metode Kernel

Catatan ini menguraikan konsep dan implementasi dari algoritma *Support Vector Machines* (SVM). SVM adalah algoritma *supervised learning* yang sangat tangguh untuk pemodelan klasifikasi dan regresi, baik pada data berdimensi rendah maupun berdimensi sangat tinggi.

Fokus utama pada modul ini adalah memahami transisi dari SVM Linier menuju SVM non-linier menggunakan teknik **Metode Kernel** (*Kernel Trick*), menganalisis pengaruh parameter kompleksitas (`C` dan `gamma`), serta memahami mengapa algoritma ini sangat sensitif terhadap skala fitur data.

#### **Tujuan Pembelajaran**
* Memahami konsep batas keputusan (hiperbidang) dan margin maksimal pada SVM Linier.
* Mengimplementasikan *Kernel Trick* (khususnya *Radial Basis Function* / RBF) untuk memproyeksikan data ke dimensi tinggi secara implisit guna menyelesaikan masalah klasifikasi non-linier.
* Mengendalikan kompleksitas model dengan menyetel hiperparameter `C` (toleransi kesalahan margin) dan `gamma` (lebar radius pengaruh data).
* Membuktikan secara empiris urgensi prapemrosesan penskalaan data (*Feature Scaling*) sebelum melatih model SVM.

In [29]:
# Memuat pustaka komputasi numerik dan visualisasi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Memuat pembuat dataset sintetis dan dataset riil
from sklearn.datasets import make_circles, load_breast_cancer
from sklearn.model_selection import train_test_split

# Memuat modul prapemrosesan dan model SVM
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import LinearSVC, SVC

# Konfigurasi visualisasi
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

print("Pustaka untuk Support Vector Machines berhasil dimuat.")

Pustaka untuk Support Vector Machines berhasil dimuat.


## 1. Linear SVM dan Limitasi pada Data Non-Linier

*LinearSVC* bekerja dengan mencari sebuah hiperbidang garis lurus yang memisahkan antar-kelas dengan jarak margin terbesar. Namun, model linier akan gagal secara absolut jika himpunan data memiliki topologi melingkar atau persilangan kompleks.

Untuk mengatasi ini, SVM menggunakan **Metode Kernel** (*Kernel Trick*). Alih-alih menambahkan fitur baru secara manual (seperti fungsi polinomial), metode kernel menghitung kemiripan antar-titik data seolah-olah data tersebut telah diproyeksikan ke ruang berdimensi sangat tinggi.

Kernel yang paling umum digunakan adalah **Radial Basis Function (RBF)** atau Kernel Gaussian, dengan rumusan matematis:
$$K(x_1, x_2) = \exp(-\gamma \|x_1 - x_2\|^2)$$
*(Di mana jarak Euclidean kuadrat antar-titik dikendalikan oleh parameter $\gamma$).*

In [30]:
# Membuat dataset sintetis berbentuk lingkaran (Non-Linier)
X_circ, y_circ = make_circles(n_samples=300, noise=0.1, factor=0.4, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_circ, y_circ, random_state=42)

# 1. Klasifikasi dengan LinearSVC (Garis Lurus)
svm_linier = LinearSVC(max_iter=10000, random_state=42).fit(X_train_c, y_train_c)

# 2. Klasifikasi dengan SVM Kernel RBF (Non-Linier)
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42).fit(X_train_c, y_train_c)

print("=== Komparasi Batas Keputusan Linier vs Non-Linier ===")
print(f"Akurasi LinearSVC (Hanya Garis Lurus) : {svm_linier.score(X_test_c, y_test_c):.4f}")
print(f"Akurasi SVC Kernel RBF (Batas Fleksibel) : {svm_rbf.score(X_test_c, y_test_c):.4f}")
print("Analisis: Data sirkuler tidak dapat dipecahkan oleh model linier, namun diselesaikan dengan sempurna oleh RBF Kernel.")

=== Komparasi Batas Keputusan Linier vs Non-Linier ===
Akurasi LinearSVC (Hanya Garis Lurus) : 0.3200
Akurasi SVC Kernel RBF (Batas Fleksibel) : 1.0000
Analisis: Data sirkuler tidak dapat dipecahkan oleh model linier, namun diselesaikan dengan sempurna oleh RBF Kernel.


## 2. Mengendalikan Kompleksitas: Parameter `C` dan `Gamma`

Pada SVM berbasis Kernel RBF, terdapat dua hiperparameter utama yang secara bersamaan mengontrol seberapa ketat model menyesuaikan diri dengan data latih:

* **Parameter `gamma`:** Menentukan seberapa jauh pengaruh satu titik data latih.
  * `gamma` kecil $\rightarrow$ Radius pengaruh sangat lebar, batas keputusan sangat mulus dan linier (*underfitting*).
  * `gamma` besar $\rightarrow$ Radius pengaruh sempit, titik data sangat dominan di area sekitarnya, batas keputusan sangat bergerigi mengikuti titik (*overfitting*).
* **Parameter `C` (Regularisasi):** Membatasi pentingnya setiap titik data.
  * `C` kecil $\rightarrow$ Penalti lemah, batas keputusan sangat terbatas (*restricted*), menghasilkan margin yang lebar dan mengabaikan kesalahan klasifikasi kecil.
  * `C` besar $\rightarrow$ Penalti kuat, model berusaha memisahkan setiap titik dengan sempurna tanpa kesalahan klasifikasi pada data latih.

In [31]:
# Memuat Dataset Kanker Payudara untuk eksperimen hiperparameter
cancer = load_breast_cancer()
X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=42
)

print("=== Pengaruh Parameter C dan Gamma pada SVM RBF ===")

# Skenario 1: C default, Gamma default (Bawaan)
svm_default = SVC(C=1.0, gamma='scale').fit(X_train_k, y_train_k)
print(f"C=1.0,  Gamma=Scale | Skor Latih: {svm_default.score(X_train_k, y_train_k):.4f} | Skor Uji: {svm_default.score(X_test_k, y_test_k):.4f}")

# Skenario 2: Gamma sangat besar (Overfitting absolut)
svm_gamma_high = SVC(C=1.0, gamma=10).fit(X_train_k, y_train_k)
print(f"C=1.0,  Gamma=10    | Skor Latih: {svm_gamma_high.score(X_train_k, y_train_k):.4f} | Skor Uji: {svm_gamma_high.score(X_test_k, y_test_k):.4f}")

# Skenario 3: C besar, Gamma kecil (Kombinasi Seimbang/Optimal)
svm_optimal = SVC(C=100.0, gamma=0.001).fit(X_train_k, y_train_k)
print(f"C=100.0, Gamma=0.001| Skor Latih: {svm_optimal.score(X_train_k, y_train_k):.4f} | Skor Uji: {svm_optimal.score(X_test_k, y_test_k):.4f}")

=== Pengaruh Parameter C dan Gamma pada SVM RBF ===
C=1.0,  Gamma=Scale | Skor Latih: 0.9178 | Skor Uji: 0.9231
C=1.0,  Gamma=10    | Skor Latih: 1.0000 | Skor Uji: 0.6294
C=100.0, Gamma=0.001| Skor Latih: 1.0000 | Skor Uji: 0.9021


## 3. Sensitivitas terhadap Penskalaan Fitur (*Feature Scaling*)

Algoritma SVM berbasis kernel mengukur jarak spasial (Euclidean) antara himpunan fitur. Jika ada satu fitur memiliki skala ribuan (misal: luas area) sedangkan fitur lainnya hanya berskala desimal (misal: tingkat kehalusan), fitur berskala masif akan mendominasi dan mengacaukan perhitungan fungsi RBF.

Oleh karena itu, **wajib** mengubah skala seluruh fitur agar seragam (berada di rentang nilai yang komparabel) sebelum melatih model SVM. Penskalaan yang paling sering direkomendasikan adalah `MinMaxScaler` (mengubah rentang fitur persis di antara 0 dan 1) atau `StandardScaler`.

In [32]:
# Evaluasi SVM pada Dataset Kanker (Tanpa Penskalaan)
svm_unscaled = SVC(C=100).fit(X_train_k, y_train_k)
print("=== Urgensi Penskalaan Data (Feature Scaling) ===")
print(f"Akurasi SVM Tanpa Penskalaan : {svm_unscaled.score(X_test_k, y_test_k):.4f}")

# Menerapkan Prapemrosesan: MinMaxScaler (Membatasi fitur antara 0 dan 1)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_k)
# HANYA gunakan fungsi transform() pada data uji untuk menghindari kebocoran data
X_test_scaled = scaler.transform(X_test_k)

# Evaluasi SVM pada Dataset Kanker (Dengan Penskalaan)
svm_scaled = SVC(C=100).fit(X_train_scaled, y_train_k)
print(f"Akurasi SVM Dengan MinMaxScaler: {svm_scaled.score(X_test_scaled, y_test_k):.4f}")
print("Analisis: Penskalaan fitur mendongkrak akurasi secara drastis dari ~93% menjadi nyaris sempurna.")

=== Urgensi Penskalaan Data (Feature Scaling) ===
Akurasi SVM Tanpa Penskalaan : 0.9441
Akurasi SVM Dengan MinMaxScaler: 0.9580
Analisis: Penskalaan fitur mendongkrak akurasi secara drastis dari ~93% menjadi nyaris sempurna.


## Kesimpulan

Support Vector Machines (SVM) memberikan kinerja yang memukau dan tangguh untuk pemodelan data berdimensi rendah maupun tinggi.

**Keunggulan SVM:**
* Sangat fleksibel berkat Metode Kernel, mampu menyesuaikan batas keputusan yang teramat rumit.
* Memiliki dasar matematika yang kokoh dalam memaksimalkan margin klasifikasi secara optimal.

**Tantangan dan Kekurangan:**
* **Sangat butuh prapemrosesan:** Model akan gagal total jika skala fitur data tidak seragam.
* **Tuning Hiperparameter yang Kritis:** Penentuan paduan parameter `C` dan `gamma` yang kurang pas akan langsung menyebabkan model memburuk secara instan (mudah memicu *overfitting* atau *underfitting* yang parah).
* Interpretasi batas keputusan (*decision boundary*) dari model SVM Kernel lebih sulit dijelaskan kepada praktisi non-teknis jika dibandingkan dengan *Decision Trees* atau *Logistic Regression*.